**kalın metin**# 🧠 O-ISAC CoT Master Pipeline (V4)

**"Optical Integrated Sensing and Communication"** Sistematik Derlemesi için Ana Yönetim Paneli.

**Aşamalar:**
1. 📦 Setup & Mount
2. 🏭 Phase 1: Data Prep (PDF → Markdown)
3. 👁️ Phase 2: Visual Analysis (Gemini Vision)
4. 🧠 Phase 3: Integrated Reasoning (V4 Llama Engine) **[NEW]**
5. 📊 Results & Export

**Gereksinimler:**
- Colab GPU Runtime (T4 veya A100)
- GROQ_API_KEY (Colab Secrets)
- GOOGLE_API_KEY (Colab Secrets)

---
**Son Güncelleme:** 2026-01-01 16:10
**Versiyon:** 4.0 (The Factory)

In [1]:
!nvidia-smi

Thu Jan  1 17:10:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 📦 Section 1: Setup & Mount

In [1]:
# @title 1.1 Install Dependencies
# Phase 1 & 2 dependencies
print('🔄 Cleaning up and installing marker-pdf...')
!pip uninstall -y marker marker-pdf -q
!pip install marker-pdf --upgrade --force-reinstall -q
!pip install transformers torch pillow -q

# Phase 3 & V4 Engine dependencies
!pip install groq nest_asyncio pandas pyyaml -q
!pip install -q -U google-generativeai

print("✅ Tüm bağımlılıklar yüklendi!")

🔄 Cleaning up and installing marker-pdf...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.45.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.0 which is incompatible.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatib

In [2]:
# @title 1.2 Mount Google Drive & Setup Paths
from google.colab import drive
from google.colab import userdata
import os
import sys

# Mount Drive
drive.mount('/content/drive')

# Project Paths
PROJECT_ROOT = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, 'analysis/nb')
COT_LAB_DIR = os.path.join(PROJECT_ROOT, 'analysis/cot_lab')
PDF_DIR = os.path.join(PROJECT_ROOT, 'data/ret_docs')
MARKDOWN_DIR = os.path.join(PROJECT_ROOT, 'data/proc_markdowns')
OUTPUT_DIR_V4 = os.path.join(PROJECT_ROOT, 'data/ext_res_v4')

# Add to Python Path
sys.path.insert(0, NOTEBOOKS_DIR)
sys.path.insert(0, PROJECT_ROOT)

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📄 PDF Directory: {PDF_DIR}")
print(f"📝 Markdown Directory: {MARKDOWN_DIR}")
print(f"📊 V4 Output Directory: {OUTPUT_DIR_V4}")
print("✅ Paths configured!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Project Root: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST
📄 PDF Directory: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST/data/ret_docs
📝 Markdown Directory: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST/data/proc_markdowns
📊 V4 Output Directory: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST/data/ext_res_v4
✅ Paths configured!


In [3]:
# @title 1.3 Load API Keys
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

    # 3-Stage Fallback Keys
    try:
        os.environ["gapi2"] = userdata.get('gapi2')
    except: pass

    # Nvidia Backup
    try:
        os.environ["nvidia/nemotron-nano-12b-v2-vl:free"] = userdata.get('nvidia/nemotron-nano-12b-v2-vl:free')
    except: pass

    print("✅ API Keys (Groq + Google + Backups) yüklendi!")
except Exception as e:
    print("❌ HATA: Sol menüden 🔑 Secrets bölümüne API anahtarlarını ekleyin!")
    print(f"   Hata detayı: {e}")

✅ API Keys (Groq + Google + Backups) yüklendi!


In [4]:
# @title 1.4 [TEST] Verify Marker v1.0 API
# Run this to confirm the API is correct before Phase 1
try:
    from marker.models import create_model_dict
    from marker.converters.pdf import PdfConverter
    print('✅ Import başarılı! Marker v1.0+ API kullanılabilir.')
    print(f'   PdfConverter: {PdfConverter}')
    print(f'   create_model_dict: {create_model_dict}')
except ImportError as e:
    print(f'❌ Import hatası: {e}')
    print('💡 Lütfen 1.1 hücresini tekrar çalıştırın ve Runtime > Restart session yapın.')


✅ Import başarılı! Marker v1.0+ API kullanılabilir.
   PdfConverter: <class 'marker.converters.pdf.PdfConverter'>
   create_model_dict: <function create_model_dict at 0x7e448c650c20>


---
## 🏭 Section 2: Phase 1 - Digitalization (PDF → Markdown)

**⚠️ GPU Gerektirir!** Bu adım PDF'leri OCR ile markdown'a çevirir.
*Engine: `marker-pdf`*

In [7]:
# @title 2.1 Import & Status Check
import extraction_pipeline_v3 as v3
from extraction_pipeline_v3 import Config
import importlib

# FORCE RELOAD V3 to pick up optimizations
importlib.reload(v3)

# Initialize
Config.init_dirs()
checkpoint = v3.CheckpointManager(Config.CHECKPOINT_FILE)

# Show status
processed_count = len(checkpoint.data.get('processed', {}))
import glob
pdf_count = len(glob.glob(os.path.join(PDF_DIR, '*.pdf')))

print(f"📊 PDF Durumu: {pdf_count} toplam, {processed_count} işlenmiş.")

Environment: Google Colab
📊 PDF Durumu: 221 toplam, 221 işlenmiş.


In [6]:
# @title 2.2 Run Digitization (Phase 1)
# ⚠️ paper başına ~2 dk sürer

FORCE_REPROCESS = False # @param {type:"boolean"}

print("⏳ Phase 1: Dijitalleştirme başlıyor...")
v3.phase1_marker_conversion(checkpoint, force_all=FORCE_REPROCESS)
print("✅ Phase 1 tamamlandı!")

⏳ Phase 1: Dijitalleştirme başlıyor...

📄 PHASE 1: PDF → MARKDOWN (Marker) - Batch Mode
Found 221 PDFs
   ⏩ O_ISAC_001 - already processed, skipping
   ⏩ O_ISAC_002 - already processed, skipping
   ⏩ O_ISAC_003 - already processed, skipping
   ⏩ O_ISAC_004 - already processed, skipping
   ⏩ O_ISAC_005 - already processed, skipping
   ⏩ O_ISAC_006 - already processed, skipping
   ⏩ O_ISAC_007 - already processed, skipping
   ⏩ O_ISAC_008 - already processed, skipping
   ⏩ O_ISAC_009 - already processed, skipping
   ⏩ O_ISAC_010 - already processed, skipping
   ⏩ O_ISAC_011 - already processed, skipping
   ⏩ O_ISAC_012 - already processed, skipping
   ⏩ O_ISAC_013 - already processed, skipping
   ⏩ O_ISAC_014 - already processed, skipping
   ⏩ O_ISAC_015 - already processed, skipping
   ⏩ O_ISAC_016 - already processed, skipping
   ⏩ O_ISAC_017 - already processed, skipping
   ⏩ O_ISAC_018 - already processed, skipping
   ⏩ O_ISAC_019 - already processed, skipping
   ⏩ O_ISAC_020 - alrea

Recognizing Text: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]
Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7e42d5c45800>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error

   ✅ Done

[2/63] 🔨 Processing: O_ISAC_161


Recognizing Text: 100%|██████████| 290/290 [01:04<00:00,  4.52it/s]


   ✅ Done

[3/63] 🔨 Processing: O_ISAC_162


...

   ✅ Done

[62/63] 🔨 Processing: O_ISAC_386


Recognizing Text: 100%|██████████| 3/3 [00:01<00:00,  1.97it/s]
Detecting bboxes: 0it [00:00, ?it/s]


   ✅ Done

[63/63] 🔨 Processing: O_ISAC_388


Recognizing Text: 100%|██████████| 187/187 [01:45<00:00,  1.77it/s]
Detecting bboxes: 0it [00:00, ?it/s]


   ✅ Done

🧹 Cleaning up Phase 1 models from GPU...
   🧹 GPU Memory Cleared

✅ Phase 1 Complete
✅ Phase 1 tamamlandı!


---
## 🖼️ Section 3: Phase 2 - Visual Analysis

Grafikleri ve şemaları anlamlandırır.
*Engine: `Gemini 2.5 Flash`*

In [ ]:
# @title 3.1 Run Visual Analysis
print("⏳ Phase 2: Görsel analiz başlıyor...")
v3.phase2_visual_analysis(checkpoint)
print("✅ Phase 2 tamamlandı!")

⏳ Phase 2: Görsel analiz başlıyor...

👁️ PHASE 2: VISUAL ANALYSIS (Local GPU + Gemini Fallback)
     ⚠️ No GPU found. Skipping Local Vision Model load.
   🔑 Using Primary Key: GOOGLE_API_KEY
Papers to analyze: 157
   ⏩ O_ISAC_001 - already analyzed, skipping
   ⏩ O_ISAC_002 - already analyzed, skipping
   ⏩ O_ISAC_003 - already analyzed, skipping
   ⏩ O_ISAC_004 - already analyzed, skipping
   ⏩ O_ISAC_005 - already analyzed, skipping
   ⏩ O_ISAC_006 - already analyzed, skipping
   ⏩ O_ISAC_007 - already analyzed, skipping
   ⏩ O_ISAC_008 - already analyzed, skipping
   ⏩ O_ISAC_009 - already analyzed, skipping
   ⏩ O_ISAC_010 - already analyzed, skipping
   ⏩ O_ISAC_011 - already analyzed, skipping
   ⏩ O_ISAC_012 - already analyzed, skipping
   ⏩ O_ISAC_013 - already analyzed, skipping
   ⏩ O_ISAC_014 - already analyzed, skipping
   ⏩ O_ISAC_015 - already analyzed, skipping
   ⏩ O_ISAC_016 - already analyzed, skipping
   ⏩ O_ISAC_017 - already analyzed, skipping
   ⏩ O_ISAC_018 - alr

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1999.99ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3010.59ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3794.09ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5840.31ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4253.77ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3387.14ms


     ⏳ Primary Key Rate Limit! Switching to Secondary Key (gapi2)...


     ⏳ Gemini Rate Limit! Switching to Backup (Nvidia)...
     ⚠️ Gemini API Limit Reached consistently. Switching to Backup Mode PERMANENTLY for this run.
     🔄 Switching to Backup Model: nvidia/nemotron-nano-12b-v2-vl:free
     ⚠️ Error preparing image for backup: function takes at most 16 arguments (17 given)
     ⚠️ Error preparing image for backup: function takes at most 16 arguments (17 given)
     ⚠️ Error preparing image for backup: function takes at most 16 arguments (17 given)
     ⚠️ Error preparing image for backup: function takes at most 16 arguments (17 given)
     📝 Nvidia Output Preview: Images provided: None   To proceed with analysis, please upload the images you'd like evaluated. Bel...
     ✅ Batch 1/1 processed (Backup Model)
     💤 Cooling down for 5.0s...
   ✅ Processed. Saved to visual_analysis.txt
[38/157] 👁️ Analyzing: O_ISAC_038
     Found 3 images -> 1 batches
     ⏳ Gemini Rate Limit! Switching to Backup (Nvidia)...
     🔄 Switching to Backup Model: nvidia

---
## 🧠 Section 4: Phase 3 - Integrated Reasoning (V4)

**YENİ:** Hem akıl yürütme (CoT) hem de veri çıkarmayı tek seferde yapar.
*Engine: `Llama 3.3 70B` + `CoTAssembler`*

In [ ]:
# @title 4.1 Import V4 Engine
import extraction_pipeline_v4 as v4
from extraction_pipeline_v4 import ConfigV4

# Init V4 environment
ConfigV4.init_dirs()
v4_checkpoint = v4.CheckpointManager(os.path.join(ConfigV4.OUTPUT_DIR, "checkpoint_v4.json"))

print("✅ V4 Engine (Factory) Hazır!")
print(f"📂 V4 Çıktı Hedefi: {ConfigV4.OUTPUT_DIR}")

In [ ]:
# @title 4.2 Run Integrated Extraction (Phase 3)

LIMIT = None # @param {type:"integer"}

print(f"🚀 Phase 3: Akıl Yürütme ve Çıkarma (Max {LIMIT} paper)...")

results = v4.phase3_integrated_extraction(v4_checkpoint, limit=LIMIT)

print(f"\n🎉 İşlem Tamamlandı! {len(results)} makale analiz edildi.")

🚀 Phase 3: Akıl Yürütme ve Çıkarma (Max 5 paper)...

🧠 PHASE 3: INTEGRATED REASONING EXTRACTION (CoTAssembler)
Papers to process: 5
[1/5] 🔨 Processing: O_ISAC_001
[INFO] Loading Recipe: analysis/cot_lab/recipes/experiment_v1_full_analysis.yaml...
[INFO] Assembling System Prompt from Modules...
[INFO] Calling Groq API (Model: llama-3.3-70b-versatile)...

[DEBUG] RAW RESPONSE LEN: 7414
[DEBUG] RAW RESPONSE START: {
  "reasoning_trace": [
       {
           "key": "step_0_visual_inspection",
           "type": "string",
           "required": true,
           "description": "MANDATORY: You MUST describe what y...
[INFO] Logging Run Evidence...
[OK] Run Logged: 20251213_121550_O_ISAC_001_llama-3.3-70b-versatile
   ✅ Success
[2/5] 🔨 Processing: O_ISAC_002
[INFO] Loading Recipe: analysis/cot_lab/recipes/experiment_v1_full_analysis.yaml...
[INFO] Assembling System Prompt from Modules...
[INFO] Calling Groq API (Model: llama-3.3-70b-versatile)...

[DEBUG] RAW RESPONSE LEN: 8391
[DEBUG] RAW RE

---
## 📊 Section 5: Results & Dashboard

In [ ]:
# @title 5.1 Show Latest Extractions
import pandas as pd
import glob

csv_path = os.path.join(ConfigV4.OUTPUT_DIR, "extraction_v4_summary.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"📊 Toplam {len(df)} kayıt bulundu.")
    display(df.head())
else:
    print("ℹ️ Henüz sonuç csv dosyası oluşmamış.")

📊 Toplam 5 kayıt bulundu.


,Paper_ID,step_0_visual_inspection,step_1_concept_analysis,step_2_benchmark_verification,step_3_strategic_critique
0,O_ISAC_001,The provided figures show the performance eval...,"The system mechanism is based on CE-OFDM, whic...","The reported metrics, such as EVM and PSD, are...",The solved problems include the mitigation of ...
1,O_ISAC_002,The provided images include diagrams of differ...,The system mechanism involves photonic teraher...,The reported metrics include a data transmissi...,The solved problems include the development of...
2,O_ISAC_003,The paper includes several diagrams and charts...,The system mechanism involves the use of visib...,The paper reports on the simulation results of...,The paper solves the problem of characterizing...
3,O_ISAC_004,"The paper includes several figures, including ...",The proposed system integrates optical fiber s...,The paper reports a salinity sensitivity of 0....,The proposed system solves the problem of inte...
4,O_ISAC_005,"The paper includes several figures and charts,...",The system mechanism involves a UAV-aided mixe...,"The paper reports various metrics, including t...",The paper solves the problem of optimizing the...
